# Machine Learning Notes
## Day 27: One-Hot Encoding — Answer Key

> **Watermark:** Amol Jagtap | amoljagtap3001@gmail.com  
> **Topic:** One-Hot Encoding for Nominal Categorical Data  
> **Difficulty:** Beginner to Intermediate  

---
### Note:
This notebook contains FULLY WORKED SOLUTIONS for every exercise in
**Day27_One_Hot_Encoding_Practice_Questions.ipynb**. Use this to check your work
or to study the reference implementation.

### Topics Covered:
1. Manual one-hot encoding by hand
2. One-Hot Encoding with pandas.get_dummies()
3. OneHotEncoder in scikit-learn (fit/transform workflow)
4. The Dummy Variable Trap — drop_first
5. Handling unseen categories at test time
6. Handling high cardinality features
7. Mini end-to-end encoding pipeline

---

In [ ]:
# ============================================================
# SETUP — Run this first!
# ============================================================
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split

print('All libraries imported successfully!')
print('Notebook by: Amol Jagtap | amoljagtap3001@gmail.com')

---
## Section 1: Manual One-Hot Encoding by Hand

In [ ]:
# ============================================================
# ANSWER 1: Manual One-Hot Encoding
# ============================================================

fruit = pd.Series(['Apple', 'Banana', 'Mango', 'Apple', 'Mango', 'Banana'], name='Fruit')

# Step 1: find unique categories
categories = sorted(fruit.unique())
print('Unique categories:', categories)

# Step 2: manually build one binary column per category
encoded_cols = {}
for cat in categories:
    col_name = f'Fruit_{cat}'
    encoded_cols[col_name] = (fruit == cat).astype(int)

# Step 3: combine into a single dataframe
manual_encoded = pd.DataFrame(encoded_cols)

# Step 4: print and verify
result = pd.concat([fruit, manual_encoded], axis=1)
print('\nManually One-Hot Encoded Result:')
print(result.to_string(index=False))

row_sums = manual_encoded.sum(axis=1)
print('\nSum across columns per row (should always be 1):')
print(row_sums.to_list())

---
## Section 2: One-Hot Encoding with pandas.get_dummies()

In [ ]:
# ============================================================
# ANSWER 2: pandas.get_dummies() on Multiple Columns
# ============================================================

df = pd.DataFrame({
    'CustomerID': [1, 2, 3, 4, 5],
    'City':       ['Mumbai', 'Delhi', 'Pune', 'Mumbai', 'Delhi'],
    'Payment':    ['Card', 'UPI', 'Cash', 'UPI', 'Card'],
    'Amount':     [500, 1200, 300, 800, 950]
})

# Encode both City and Payment, output as clean integers
df_encoded = pd.get_dummies(df, columns=['City', 'Payment'], dtype=int)

print('Encoded Data:')
print(df_encoded.to_string(index=False))

print('\nShape before encoding:', df.shape)
print('Shape after encoding: ', df_encoded.shape)

print('\nCustomerID and Amount remain unchanged numeric columns —')
print('only City (3 categories) and Payment (3 categories) were')
print('expanded into 6 new binary columns total.')

---
## Section 3: OneHotEncoder in Scikit-learn

In [ ]:
# ============================================================
# ANSWER 3: Fit/Transform Workflow with OneHotEncoder
# ============================================================

df_color = pd.DataFrame({
    'Color': ['Red', 'Blue', 'Green', 'Red', 'Blue', 'Green', 'Red', 'Blue']
})

# Step 1: train-test split
X_train, X_test = train_test_split(df_color, test_size=0.2, random_state=42)

# Step 2: create encoder
encoder = OneHotEncoder(sparse_output=False)

# Step 3: fit on training data only
encoder.fit(X_train[['Color']])

# Step 4: transform both sets
X_train_enc = encoder.transform(X_train[['Color']])
X_test_enc  = encoder.transform(X_test[['Color']])

# Step 5: print results
print('Generated feature names:', encoder.get_feature_names_out(['Color']))

print('\nX_train original:')
print(X_train['Color'].tolist())
print('X_train encoded:')
print(X_train_enc)

print('\nX_test original:')
print(X_test['Color'].tolist())
print('X_test encoded:')
print(X_test_enc)

print('\nBoth train and test arrays use the SAME 3 columns')
print('(Blue, Green, Red) because the encoder was fit only once,')
print('on training data, ensuring consistent column structure.')

---
## Section 4: The Dummy Variable Trap — drop_first

In [ ]:
# ============================================================
# ANSWER 4: Dummy Variable Trap — Demonstration and Fix
# ============================================================

df_size = pd.DataFrame({
    'Size': ['Small', 'Medium', 'Large', 'Small', 'Large']
})

# 1. WITHOUT dropping any column
full_encoded = pd.get_dummies(df_size, columns=['Size'], dtype=int)
print('Without drop_first (all k columns kept):')
print(full_encoded.to_string(index=False))
print('Number of columns created:', full_encoded.shape[1])

# 2. WITH drop_first=True
reduced_encoded = pd.get_dummies(df_size, columns=['Size'],
                                   drop_first=True, dtype=int)
print('\nWith drop_first=True (k-1 columns kept):')
print(reduced_encoded.to_string(index=False))
print('Number of columns created:', reduced_encoded.shape[1])

# 3. Explanation
print('\nExplanation: "Large" was alphabetically first, so it was dropped.')
print('If Size_Medium=0 AND Size_Small=0 for a row, that row MUST be')
print('"Large" — the dropped category is still fully recoverable from')
print('the remaining columns. No information is actually lost; we only')
print('removed the mathematically redundant column.')

---
## Section 5: Handling Unseen Categories at Test Time

In [ ]:
# ============================================================
# ANSWER 5: Handling Unseen Categories
# ============================================================

train_data = pd.DataFrame({'Brand': ['Nike', 'Adidas', 'Puma', 'Nike']})
test_data  = pd.DataFrame({'Brand': ['Adidas', 'Reebok']})  # 'Reebok' unseen

# 1. Fit a default OneHotEncoder
encoder_strict = OneHotEncoder(sparse_output=False)
encoder_strict.fit(train_data[['Brand']])

# 2. Try transforming test data with the unseen category
print('Attempting transform WITHOUT handle_unknown protection:')
try:
    encoder_strict.transform(test_data[['Brand']])
except ValueError as e:
    print(f'  ERROR raised as expected: {e}')

# 3. Create a safe encoder
encoder_safe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoder_safe.fit(train_data[['Brand']])

# 4. Transform safely
safe_result = encoder_safe.transform(test_data[['Brand']])
print('\nSafe transform WITH handle_unknown="ignore":')
print('Feature names:', encoder_safe.get_feature_names_out(['Brand']))
result_df = pd.DataFrame(safe_result,
                          columns=encoder_safe.get_feature_names_out(['Brand']))
print(pd.concat([test_data.reset_index(drop=True), result_df], axis=1).to_string(index=False))

print('\n"Reebok" (unseen) produced an all-zero row instead of raising')
print('an error, while "Adidas" was correctly encoded with a 1.')

---
## Section 6: Handling High Cardinality Features

In [ ]:
# ============================================================
# ANSWER 6: Reducing Cardinality Before One-Hot Encoding
# ============================================================

np.random.seed(0)
cities_pool = ['Mumbai', 'Delhi', 'Pune', 'Chennai', 'Kolkata', 'Jaipur',
               'Indore', 'Nagpur', 'Surat', 'Patna', 'Bhopal', 'Lucknow']
weights = [0.30, 0.25, 0.15, 0.08, 0.05, 0.04, 0.03, 0.03, 0.02, 0.02, 0.02, 0.01]
city_data = pd.DataFrame({
    'City': np.random.choice(cities_pool, size=200, p=weights)
})

# Encoding WITHOUT any reduction (baseline)
full_encoded = pd.get_dummies(city_data, columns=['City'], dtype=int)
cols_before = full_encoded.shape[1]

# Step 1: find top 4 most frequent cities
top_cities = city_data['City'].value_counts().nlargest(4).index
print('Top 4 cities:', list(top_cities))

# Step 2: replace everything else with 'Other'
city_data_reduced = city_data.copy()
city_data_reduced['City'] = city_data_reduced['City'].where(
    city_data_reduced['City'].isin(top_cities), other='Other')

print('\nNew value counts after grouping rare cities:')
print(city_data_reduced['City'].value_counts())

# Step 3: one-hot encode the reduced column
reduced_encoded = pd.get_dummies(city_data_reduced, columns=['City'], dtype=int)
cols_after = reduced_encoded.shape[1]

# Step 4: compare column counts
print(f'\nColumns created WITHOUT reduction: {cols_before}')
print(f'Columns created WITH reduction (Top 4 + Other): {cols_after}')
print(f'\nReduction saved {cols_before - cols_after} columns while keeping')
print('the vast majority of the data correctly represented.')

---
## Section 7: Mini End-to-End Encoding Pipeline

In [ ]:
# ============================================================
# ANSWER 7: Full One-Hot Encoding Pipeline
# ============================================================

raw = pd.DataFrame({
    'City':     ['Mumbai', 'Delhi', 'Pune', 'Mumbai', 'Delhi',
                  'Pune', 'Mumbai', 'Delhi'],
    'Payment':  ['Card', 'UPI', 'Cash', 'UPI', 'Card',
                  'Cash', 'UPI', 'Card'],
    'Purchased':[1, 0, 1, 1, 0, 0, 1, 1]
})

# Step 1: split into X and y
X = raw[['City', 'Payment']]
y = raw['Purchased']

# Step 2: train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42)

# Step 3: create encoder — handle_unknown + drop first column
encoder = OneHotEncoder(sparse_output=False,
                          handle_unknown='ignore',
                          drop='first')

# Step 4: fit on training data ONLY, then transform both sets
encoder.fit(X_train)
X_train_enc = encoder.transform(X_train)
X_test_enc  = encoder.transform(X_test)

# Step 5: print results
feature_names = encoder.get_feature_names_out(['City', 'Payment'])
print('Generated feature names (after dropping first category each):')
print(feature_names)

print('\nX_train encoded:')
print(pd.DataFrame(X_train_enc, columns=feature_names))

print('\nX_test encoded:')
print(pd.DataFrame(X_test_enc, columns=feature_names))

print('\nPipeline complete: City and Payment are now numeric, the dummy')
print('variable trap is avoided via drop="first", and any unseen category')
print('at test time would safely become an all-zero row instead of erroring.')

print('\nAmol Jagtap | amoljagtap3001@gmail.com')

---
## Summary & Quick Revision

| Concept | What You Learned |
|---|---|
| One-Hot Encoding | 1 categorical column -> k binary columns (k = #categories) |
| Best for | Nominal data — no natural order between categories |
| pandas method | `pd.get_dummies(df, columns=[...], dtype=int)` |
| sklearn class | `OneHotEncoder()` — supports fit/transform safely |
| Dummy Variable Trap | Keeping all k columns causes multicollinearity in linear models |
| Fix | `drop_first=True` (pandas) or `drop='first'` (sklearn) |
| Unseen categories | `handle_unknown='ignore'` avoids errors at test time |
| High cardinality | Group rare categories as 'Other' before encoding |
| Fit rule | Always fit on training data only, transform on test data |

---
> **Notebook by:** Amol Jagtap | amoljagtap3001@gmail.com  
> **Topic:** Day 27 — One-Hot Encoding